##### **Phare Data - Migration Cross Region**
Notebook used for assessing tenant wide items and populate a Lakehouse.
Once the data has been saved to the Lakehouse a new semantic model and report will be created.
The process makes use of semantic link and semantic link labs.

##### **Prerequisites**
- Tenant admin will be required to identify all capacities, workspaces and items.
- XMLA Read/Write will need to be enabled on the capacity.

In [ ]:
%pip install semantic-link-labs==0.14.1

In [ ]:
item_name ='assessment'
lakehouse = f"lh_{item_name}"
semantic_model_name = f"sm_{item_name}"
report_name = f"rep_{item_name}"

In [ ]:
import sempy
import sempy.fabric as fabric
import sempy_labs as labs
import sempy_labs.admin as labs_admin
import sempy_labs.report as labs_report
import pandas as pd
import json

bimjson = {'compatibilityLevel': 1604,
 'model': {'annotations': [{'name': '__PBI_TimeIntelligenceEnabled',
    'value': '0'},
   {'name': 'PBIDesktopVersion',
    'value': '2.146.7727.7 (Main)+bc5f9142edb5be5beb4b90d87b320b00b8f2d382'},
   {'name': 'PBI_QueryOrder', 'value': '["DatabaseQuery"]'},
   {'name': 'PBI_ProTooling',
    'value': '["RemoteModeling", "DevMode", "SLL", "WebModelingEdit", "SLL_UpdateDLConnection_SQL"]'},
   {'name': 'TabularEditor_SerializeOptions',
    'value': '{"IgnoreInferredObjects":True,"IgnoreInferredProperties":True,"IgnoreTimestamps":True,"SplitMultilineStrings":True,"PrefixFilenames":False,"LocalTranslations":True,"LocalPerspectives":True,"LocalRelationships":True,"Levels":["Data Sources","Perspectives","Relationships","Roles","Shared Expressions","Tables","Tables/Calculation Items","Tables/Columns","Tables/Hierarchies","Tables/Measures","Tables/Partitions","Translations"]}'},
   {'name': '__TEdtr', 'value': '1'}],
  'collation': 'Latin1_General_100_BIN2_UTF8',
  'culture': 'en-US',
  'cultures': [{'name': 'en-US',
    'linguisticMetadata': {'content': {'Language': 'en-US',
      'Version': '1.0.0'},
     'contentType': 'json'}}],
  'dataAccessOptions': {'legacyRedirects': True,
   'returnErrorValuesAsNull': True},
  'defaultPowerBIDataSourceVersion': 'powerBI_V3',
  'expressions': [{'name': 'DatabaseQuery',
    'annotations': [{'name': 'PBI_IncludeFutureArtifacts', 'value': 'False'}],
    'expression': ['let',
     '\tdatabase = Sql.Database("vxl72uat6j5e3ifax5xhxupvdy-ufczssi2ahaedkldjork26beiu.datawarehouse.fabric.microsoft.com", "adbdbcbf-c7e2-41b1-87cd-fc2aa5ffc74e")',
     'in',
     '\tdatabase'],
    'kind': 'm',
    'lineageTag': '9b99fd62-abc8-4b7b-9ebd-e696581f8fee'}],
  'relationships': [{'name': '76f9977d-9c44-959a-e1f0-631cd174a194',
    'fromColumn': 'Capacity_Id',
    'fromTable': 'Workspaces',
    'toColumn': 'Capacity_Id',
    'toTable': 'Capacities'},
   {'name': 'eb5cd574-9c3c-59f6-785d-f297244c6108',
    'fromColumn': 'Workspace_Id',
    'fromTable': 'WorkspaceItems',
    'relyOnReferentialIntegrity': True,
    'toColumn': 'Id',
    'toTable': 'Workspaces'},
   {'name': '704f3d48-adc4-5c39-1ffb-f626e170f8e8',
    'fromColumn': 'Workspace_Id',
    'fromTable': 'SemanticModels',
    'toColumn': 'Id',
    'toTable': 'Workspaces'}],
  'sourceQueryCulture': 'en-US',
  'tables': [{'name': 'Workspaces',
    'annotations': [{'name': 'PBI_ResultType', 'value': 'Table'}],
    'columns': [{'name': 'Id',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': 'f5095b8a-425a-4282-8906-4aa318949fdb',
      'sourceColumn': 'Id',
      'sourceLineageTag': 'Id',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Capacity_Id',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': '88954739-5272-4299-bef8-b206d2e0fed5',
      'sourceColumn': 'Capacity_Id',
      'sourceLineageTag': 'Capacity_Id',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Type',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': 'fc3bc44b-8cd0-4aaa-9806-17be6b641f0d',
      'sourceColumn': 'Type',
      'sourceLineageTag': 'Type',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Name',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': 'c3d67c6f-7f0e-4bdb-8af9-38f97e000108',
      'sourceColumn': 'Name',
      'sourceLineageTag': 'Name',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'State',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': '49107ca3-67eb-4c85-bfa4-b1df21fee139',
      'sourceColumn': 'State',
      'sourceLineageTag': 'State',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'}],
    'lineageTag': 'dad1aa84-1ef4-49ae-901e-7dc8e29d3d7e',
    'measures': [{'name': 'Total Workspaces',
      'changedProperties': [{'property': 'Name'}],
      'expression': 'COUNTROWS(Workspaces)',
      'formatString': '0',
      'lineageTag': '68a65d8d-7be5-4b0a-a0bd-93b13ee757c8'},
     {'name': 'Cross region readiness',
      'changedProperties': [{'property': 'Name'}],
      'expression': ['',
       'IF(',
       '    Workspaces[Total Workspaces] > 0  ,',
       '    SWITCH(',
       '    TRUE(),',
       '    ([Total Non PBI Items] + [Large Models]) <> 0, "Fabric item or Large model",',
       '    ([Total Dataflow Datamart]) <> 0, "Dataflow or Datamart",',
       '    "Good to move"',
       '    )',
       ')',
       '// IF(',
       '//     Workspaces[Total Workspaces] > 0  ,',
       '//     SWITCH(',
       '//     TRUE(),',
       '//     ([Total Non PBI Items] + [Large Models]) <> 0, "🔴",',
       '//     ([Total Dataflow Datamart]) <> 0, "🟡",',
       '//     "🟢"',
       '//     )',
       '// )'],
      'lineageTag': '594996d0-3724-464a-924c-7c3247797c6a'}],
    'partitions': [{'name': 'Workspaces',
      'mode': 'directLake',
      'source': {'entityName': 'Workspaces',
       'expressionSource': 'DatabaseQuery',
       'schemaName': 'dbo',
       'type': 'entity'}}],
    'sourceLineageTag': '[dbo].[Workspaces]'},
   {'name': 'WorkspaceItems',
    'annotations': [{'name': 'PBI_ResultType', 'value': 'Table'}],
    'columns': [{'name': 'Item_Id',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': '05a295c6-d0fe-4691-9d69-45e7f4e68b0c',
      'sourceColumn': 'Item_Id',
      'sourceLineageTag': 'Item_Id',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Item_Name',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': '7a239119-2306-4943-a47e-8985d24c9fde',
      'sourceColumn': 'Item_Name',
      'sourceLineageTag': 'Item_Name',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Type',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': 'fa2ab9b7-9f58-4cd9-9de6-89a769f554ba',
      'sourceColumn': 'Type',
      'sourceLineageTag': 'Type',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Description',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': '634b0e42-247a-4ae1-ad34-7a6c3402878e',
      'sourceColumn': 'Description',
      'sourceLineageTag': 'Description',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'State',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': '0223b582-486e-4c1f-b407-40aeffb86d1c',
      'sourceColumn': 'State',
      'sourceLineageTag': 'State',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Last_Updated_Date',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': '4997f0fe-fe24-4c0f-a48b-72709ce28448',
      'sourceColumn': 'Last_Updated_Date',
      'sourceLineageTag': 'Last_Updated_Date',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Creator_Principal_Id',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': 'dc0b4e00-223f-4567-b058-3795344fecf7',
      'sourceColumn': 'Creator_Principal_Id',
      'sourceLineageTag': 'Creator_Principal_Id',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Creator_Principal_Display_Name',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': '492896e8-e06c-4c65-bae7-11a101836607',
      'sourceColumn': 'Creator_Principal_Display_Name',
      'sourceLineageTag': 'Creator_Principal_Display_Name',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Creator_Principal_Type',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': 'a6af9883-2493-405e-980d-5858fde6c033',
      'sourceColumn': 'Creator_Principal_Type',
      'sourceLineageTag': 'Creator_Principal_Type',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Creator_User_Principal_Name',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': '99e6803b-f780-4a60-8a8e-eabf9933478d',
      'sourceColumn': 'Creator_User_Principal_Name',
      'sourceLineageTag': 'Creator_User_Principal_Name',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Workspace_Id',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': '8dc1099d-2216-440f-b298-a86825e7e27a',
      'sourceColumn': 'Workspace_Id',
      'sourceLineageTag': 'Workspace_Id',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Capacity_Id',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': 'b7f2f537-49eb-458d-9b62-affb3e1bf7e8',
      'sourceColumn': 'Capacity_Id',
      'sourceLineageTag': 'Capacity_Id',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'}],
    'lineageTag': '02ff19b6-bacd-4c72-9518-e7315543550b',
    'measures': [{'name': 'Total Items',
      'changedProperties': [{'property': 'Name'}],
      'expression': 'COUNTROWS(WorkspaceItems)',
      'formatString': '0',
      'lineageTag': '55325202-738c-41cc-874f-f7b09933ef55'},
     {'name': 'Total PBI Items',
      'changedProperties': [{'property': 'Name'}],
      'expression': 'CALCULATE(COUNTROWS(WorkspaceItems), KEEPFILTERS(WorkspaceItems[Type] IN { "Dashboard", "PaginatedReport", "Datamart", "Report" ,"SemanticModel","App", "Dataflow" }))',
      'formatString': '0',
      'lineageTag': '3e32c7a9-2855-4ba4-a8d0-5ed6d635bd0d'},
     {'name': 'Total Non PBI Items',
      'changedProperties': [{'property': 'Name'}],
      'expression': '[Total Items] - [Total PBI Items]',
      'formatString': '0',
      'lineageTag': 'ab55750d-350e-47fd-ae14-ba64376a5253'},
     {'name': 'Total Dataflow Datamart',
      'changedProperties': [{'property': 'Name'}],
      'expression': 'CALCULATE(COUNTROWS(WorkspaceItems), WorkspaceItems[Type] IN { "Dataflow", "Datamart" })',
      'formatString': '0',
      'lineageTag': '05665bea-5ff8-4e80-9f68-2b0f96beabd7'},
     {'name': 'Total Dataflow',
      'changedProperties': [{'property': 'Name'}],
      'expression': 'CALCULATE(COUNTROWS(WorkspaceItems), WorkspaceItems[Type] IN { "Dataflow" })',
      'formatString': '0',
      'lineageTag': '53c878d6-184c-4093-b2e0-0605dbd83056'}],
    'partitions': [{'name': 'WorkspaceItems',
      'mode': 'directLake',
      'source': {'entityName': 'WorkspaceItems',
       'expressionSource': 'DatabaseQuery',
       'schemaName': 'dbo',
       'type': 'entity'}}],
    'sourceLineageTag': '[dbo].[WorkspaceItems]'},
   {'name': 'Capacities',
    'annotations': [{'name': 'PBI_ResultType', 'value': 'Table'}],
    'columns': [{'name': 'Capacity_Id',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': '893cef6a-4438-40e2-8a04-1b6e1f152865',
      'sourceColumn': 'Capacity_Id',
      'sourceLineageTag': 'Capacity_Id',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Capacity_Name',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': 'c7969fa1-8906-415b-88d2-37ef730842a7',
      'sourceColumn': 'Capacity_Name',
      'sourceLineageTag': 'Capacity_Name',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Sku',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': '3437c61a-7551-487d-9aa9-4c4ce1c42a1f',
      'sourceColumn': 'Sku',
      'sourceLineageTag': 'Sku',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Region',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': '11d1d6e2-9277-4258-a915-1f25ab3eaf89',
      'sourceColumn': 'Region',
      'sourceLineageTag': 'Region',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'State',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': '2b02f14a-2d01-4252-9556-fe4ba0961345',
      'sourceColumn': 'State',
      'sourceLineageTag': 'State',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'}],
    'lineageTag': '6382af07-8377-43ab-97fa-5422e0da102a',
    'measures': [{'name': 'Total Capacities',
      'changedProperties': [{'property': 'Name'}],
      'expression': 'COUNTROWS(Capacities)',
      'formatString': '0',
      'lineageTag': '0e26151c-257d-4051-8351-d593a6c52de0'},
     {'name': 'Selected Capacity',
      'changedProperties': [{'property': 'Name'}],
      'expression': 'SELECTEDVALUE(Capacities[Capacity_Name])',
      'lineageTag': '9d57a129-d9a4-480d-89b6-9d0f5a86cb2b'},
     {'name': 'Single Capacity Selected',
      'changedProperties': [{'property': 'Name'}],
      'expression': 'IF(HASONEVALUE(Capacities[Capacity_Name]),"","Please select a single capacity to see size and region.")',
      'lineageTag': '66f870a5-16a5-40ed-81ad-37bbe53e2b91'},
     {'name': 'Tansparent colour',
      'changedProperties': [{'property': 'Name'}],
      'expression': 'IF(HASONEVALUE(Capacities[Capacity_Name]),"#FFFFFF00","#FFFFFF")',
      'lineageTag': 'd9eb2daf-4dc3-429e-9545-fd53ea861dc8'}],
    'partitions': [{'name': 'Capacities',
      'mode': 'directLake',
      'source': {'entityName': 'Capacities',
       'expressionSource': 'DatabaseQuery',
       'schemaName': 'dbo',
       'type': 'entity'}}],
    'sourceLineageTag': '[dbo].[Capacities]'},
   {'name': 'SemanticModels',
    'annotations': [{'name': 'PBI_ResultType', 'value': 'Table'}],
    'columns': [{'name': 'Dataset_Id',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': '404d6a08-160d-4f77-816b-4d972a80fc47',
      'sourceColumn': 'Dataset_Id',
      'sourceLineageTag': 'Dataset_Id',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Dataset_Name',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': 'ee58bd50-b319-4860-ace7-1760ea6186d4',
      'sourceColumn': 'Dataset_Name',
      'sourceLineageTag': 'Dataset_Name',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Web_URL',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': '62a988ce-01cd-4895-8459-e9d0a262ca39',
      'sourceColumn': 'Web_URL',
      'sourceLineageTag': 'Web_URL',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Add_Rows_API_Enabled',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'boolean',
      'formatString': '"TRUE";"TRUE";"FALSE"',
      'lineageTag': '2fe675e0-55f3-4a43-b66d-583db8429de0',
      'sourceColumn': 'Add_Rows_API_Enabled',
      'sourceLineageTag': 'Add_Rows_API_Enabled',
      'sourceProviderType': 'bit',
      'summarizeBy': 'none'},
     {'name': 'Configured_By',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': 'aaebd6dd-ca0b-4ddb-900e-ca807033d59d',
      'sourceColumn': 'Configured_By',
      'sourceLineageTag': 'Configured_By',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Is_Refreshable',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'boolean',
      'formatString': '"TRUE";"TRUE";"FALSE"',
      'lineageTag': 'adfb7d4b-996f-44e0-8ac8-63e39ac71bfe',
      'sourceColumn': 'Is_Refreshable',
      'sourceLineageTag': 'Is_Refreshable',
      'sourceProviderType': 'bit',
      'summarizeBy': 'none'},
     {'name': 'Is_Effective_Identity_Required',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'boolean',
      'formatString': '"TRUE";"TRUE";"FALSE"',
      'lineageTag': '536beb07-efc0-46a1-a5cf-c2e03c19ba63',
      'sourceColumn': 'Is_Effective_Identity_Required',
      'sourceLineageTag': 'Is_Effective_Identity_Required',
      'sourceProviderType': 'bit',
      'summarizeBy': 'none'},
     {'name': 'Is_Effective_Identity_Roles_Required',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'boolean',
      'formatString': '"TRUE";"TRUE";"FALSE"',
      'lineageTag': 'c7665754-1994-4dc4-9883-fc08b5dd88e0',
      'sourceColumn': 'Is_Effective_Identity_Roles_Required',
      'sourceLineageTag': 'Is_Effective_Identity_Roles_Required',
      'sourceProviderType': 'bit',
      'summarizeBy': 'none'},
     {'name': 'Target_Storage_Mode',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': 'a2a6e011-fd40-43a5-84b9-07ec39459624',
      'sourceColumn': 'Target_Storage_Mode',
      'sourceLineageTag': 'Target_Storage_Mode',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Created_Date',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'dateTime',
      'formatString': 'General Date',
      'lineageTag': 'f0bab76a-e671-44f1-878e-221a39b1ad6a',
      'sourceColumn': 'Created_Date',
      'sourceLineageTag': 'Created_Date',
      'sourceProviderType': 'datetime2',
      'summarizeBy': 'none'},
     {'name': 'Content_Provider_Type',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': '0ac253fc-36bf-4b9c-9671-5764a3c0651b',
      'sourceColumn': 'Content_Provider_Type',
      'sourceLineageTag': 'Content_Provider_Type',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Create_Report_Embed_URL',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': '18b523ef-9a2f-4d8a-87c7-a3f07ca5f328',
      'sourceColumn': 'Create_Report_Embed_URL',
      'sourceLineageTag': 'Create_Report_Embed_URL',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'QnA_Embed_URL',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': '0544e77c-484d-4fe2-a3b3-8a0a48f1276c',
      'sourceColumn': 'QnA_Embed_URL',
      'sourceLineageTag': 'QnA_Embed_URL',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Upstream_Datasets',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': '8a8779b6-fd3a-44b2-b550-fac19465638f',
      'sourceColumn': 'Upstream_Datasets',
      'sourceLineageTag': 'Upstream_Datasets',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Users',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': 'b4ad45cc-06f7-4196-b760-254f2513f6cd',
      'sourceColumn': 'Users',
      'sourceLineageTag': 'Users',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Is_In_Place_Sharing_Enabled',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'boolean',
      'formatString': '"TRUE";"TRUE";"FALSE"',
      'lineageTag': 'f8b2c617-9c29-445c-88d7-0a265f21999f',
      'sourceColumn': 'Is_In_Place_Sharing_Enabled',
      'sourceLineageTag': 'Is_In_Place_Sharing_Enabled',
      'sourceProviderType': 'bit',
      'summarizeBy': 'none'},
     {'name': 'Workspace_Id',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'string',
      'lineageTag': 'b9b9beb3-ba0b-4996-b4b1-3877e572bebc',
      'sourceColumn': 'Workspace_Id',
      'sourceLineageTag': 'Workspace_Id',
      'sourceProviderType': 'varchar(8000)',
      'summarizeBy': 'none'},
     {'name': 'Auto_Sync_Read_Only_Replicas',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'boolean',
      'formatString': '"TRUE";"TRUE";"FALSE"',
      'lineageTag': 'b26c129f-2364-4561-9f75-fc313ac8e999',
      'sourceColumn': 'Auto_Sync_Read_Only_Replicas',
      'sourceLineageTag': 'Auto_Sync_Read_Only_Replicas',
      'sourceProviderType': 'bit',
      'summarizeBy': 'none'},
     {'name': 'Max_Read_Only_Replicas',
      'annotations': [{'name': 'SummarizationSetBy', 'value': 'Automatic'}],
      'dataType': 'int64',
      'formatString': '0',
      'lineageTag': '936012c7-1d89-4605-81f3-75b9f556dfe8',
      'sourceColumn': 'Max_Read_Only_Replicas',
      'sourceLineageTag': 'Max_Read_Only_Replicas',
      'sourceProviderType': 'bigint',
      'summarizeBy': 'sum'}],
    'lineageTag': 'fc54bd41-11d5-4d78-b14c-b1bcf0bc0b7f',
    'measures': [{'name': 'Large Models',
      'changedProperties': [{'property': 'Name'}],
      'expression': 'CALCULATE(COUNTROWS(SemanticModels),SemanticModels[Target_Storage_Mode]="PremiumFiles")',
      'formatString': '0',
      'lineageTag': 'e900cf3c-d14b-41d4-9777-720fb72f4930'},
     {'name': 'DF + LSM',
      'changedProperties': [{'property': 'Name'}],
      'expression': ['',
       'IF(',
       '    Workspaces[Total Workspaces] = 1  ,',
       '    IF([Total Dataflow]>0 && [Large Models]>0, "Y")',
       ')'],
      'lineageTag': 'c3f70cc4-0235-4ae4-bdf1-b69796e5b50e'}],
    'partitions': [{'name': 'SemanticModels',
      'mode': 'directLake',
      'source': {'entityName': 'SemanticModels',
       'expressionSource': 'DatabaseQuery',
       'schemaName': 'dbo',
       'type': 'entity'}}],
    'sourceLineageTag': '[dbo].[SemanticModels]'}]}}


reportjson = {'config': '{"version":"5.57","themeCollection":{"baseTheme":{"name":"CY24SU08","type":2,"version":"5.58"}},"activeSectionIndex":0,"defaultDrillFilterOtherVisuals":true,"linguisticSchemaSyncVersion":0,"settings":{"useNewFilterPaneExperience":true,"allowChangeFilterTypes":true,"useStylableVisualContainerHeader":true,"queryLimitOption":6,"useEnhancedTooltips":true,"exportDataMode":1,"useDefaultAggregateDisplayName":true},"objects":{"section":[{"properties":{"verticalAlignment":{"expr":{"Literal":{"Value":"\'Top\'"}}}}}],"outspacePane":[{"properties":{"expanded":{"expr":{"Literal":{"Value":"false"}}}}}]}}',
 'layoutOptimization': 0,
 'resourcePackages': [{'resourcePackage': {'disabled': False,
    'items': [{'name': 'CY24SU08',
      'path': 'BaseThemes/CY24SU08.json',
      'type': 202}],
    'name': 'SharedResources',
    'type': 2}}],
 'sections': [{'config': '{"relationships":[{"source":"c907bd0902a1828474bd","target":"6f2e1e9f95e4fb32a377","type":3},{"source":"c907bd0902a1828474bd","target":"eeb0d5b9b6bc0f1f96a1","type":3},{"source":"c907bd0902a1828474bd","target":"eec2319a0fbda582c675","type":3},{"source":"c907bd0902a1828474bd","target":"9ca366c185006c4a97c8","type":3},{"source":"c907bd0902a1828474bd","target":"9c67315f6a6dc100ea50","type":3},{"source":"c907bd0902a1828474bd","target":"1000368e487751cf6365","type":3},{"source":"eeb0d5b9b6bc0f1f96a1","target":"eec2319a0fbda582c675","type":3},{"source":"eeb0d5b9b6bc0f1f96a1","target":"89a6011ae737ee39ad7a","type":3},{"source":"eeb0d5b9b6bc0f1f96a1","target":"b092367cb9c80420876d","type":3},{"source":"eeb0d5b9b6bc0f1f96a1","target":"9ca366c185006c4a97c8","type":3},{"source":"9c67315f6a6dc100ea50","target":"eec2319a0fbda582c675","type":3},{"source":"9c67315f6a6dc100ea50","target":"9ca366c185006c4a97c8","type":3},{"source":"9c67315f6a6dc100ea50","target":"b092367cb9c80420876d","type":3},{"source":"9c67315f6a6dc100ea50","target":"89a6011ae737ee39ad7a","type":3},{"source":"1000368e487751cf6365","target":"89a6011ae737ee39ad7a","type":3},{"source":"1000368e487751cf6365","target":"b092367cb9c80420876d","type":3},{"source":"1000368e487751cf6365","target":"eec2319a0fbda582c675","type":3},{"source":"1000368e487751cf6365","target":"9c67315f6a6dc100ea50","type":1}],"objects":{"background":[{"properties":{"image":{"image":{"name":{"expr":{"Literal":{"Value":"\'\'"}}},"url":{"expr":{"Literal":{"Value":"\'\'"}}}}},"transparency":{"expr":{"Literal":{"Value":"0D"}}},"color":{"solid":{"color":{"expr":{"ThemeDataColor":{"ColorId":0,"Percent":-0.1}}}}}}}]}}',
   'displayName': 'Summary',
   'displayOption': 1,
   'filters': '[]',
   'height': 720.0,
   'name': 'b8e9e4595d636e98d084',
   'visualContainers': [{'config': '{"name":"1000368e487751cf6365","layouts":[{"id":0,"position":{"x":444,"y":489.16966998798677,"z":5000,"width":558,"height":214.1926365627871,"tabOrder":3000}}],"singleVisual":{"visualType":"tableEx","projections":{"Values":[{"queryRef":"SemanticModels.Dataset_Name"},{"queryRef":"SemanticModels.Target_Storage_Mode"},{"queryRef":"SemanticModels.Dataset_Id"}]},"prototypeQuery":{"Version":2,"From":[{"Name":"s","Entity":"SemanticModels","Type":0}],"Select":[{"Column":{"Expression":{"SourceRef":{"Source":"s"}},"Property":"Target_Storage_Mode"},"Name":"SemanticModels.Target_Storage_Mode","NativeReferenceName":"Storage_Mode"},{"Column":{"Expression":{"SourceRef":{"Source":"s"}},"Property":"Dataset_Name"},"Name":"SemanticModels.Dataset_Name","NativeReferenceName":"Dataset_Name"},{"Column":{"Expression":{"SourceRef":{"Source":"s"}},"Property":"Dataset_Id"},"Name":"SemanticModels.Dataset_Id","NativeReferenceName":"Dataset_Id"}]},"columnProperties":{"SemanticModels.Target_Storage_Mode":{"displayName":"Storage_Mode"}},"drillFilterOtherVisuals":true,"objects":{"total":[{"properties":{}}]},"vcObjects":{"title":[{"properties":{"show":{"expr":{"Literal":{"Value":"true"}}},"text":{"expr":{"Literal":{"Value":"\'Semantic Models\'"}}}}}],"background":[{"properties":{"transparency":{"expr":{"Literal":{"Value":"100D"}}}}}],"divider":[{"properties":{"show":{"expr":{"Literal":{"Value":"true"}}},"style":{"expr":{"Literal":{"Value":"\'dashed\'"}}},"width":{"expr":{"Literal":{"Value":"2D"}}}}}]}}}',
     'filters': '[]',
     'height': 214.19,
     'width': 558.0,
     'x': 444.0,
     'y': 489.17,
     'z': 5000.0},
    {'config': '{"name":"2e2af0a2fc39e94fdbe7","layouts":[{"id":0,"position":{"x":612.7689532274387,"y":485.40913157624556,"z":12000,"width":297.9739223537349,"height":27.63467828280606,"tabOrder":12000}}],"singleVisual":{"visualType":"slicer","projections":{"Values":[{"queryRef":"SemanticModels.Target_Storage_Mode","active":true}]},"prototypeQuery":{"Version":2,"From":[{"Name":"s","Entity":"SemanticModels","Type":0}],"Select":[{"Column":{"Expression":{"SourceRef":{"Source":"s"}},"Property":"Target_Storage_Mode"},"Name":"SemanticModels.Target_Storage_Mode","NativeReferenceName":"Target_Storage_Mode"}]},"drillFilterOtherVisuals":true,"objects":{"data":[{"properties":{"mode":{"expr":{"Literal":{"Value":"\'Basic\'"}}}}}],"general":[{"properties":{"orientation":{"expr":{"Literal":{"Value":"1D"}}},"responsive":{"expr":{"Literal":{"Value":"false"}}}}}],"header":[{"properties":{"show":{"expr":{"Literal":{"Value":"false"}}}}}]},"vcObjects":{"visualHeader":[{"properties":{"background":{"solid":{"color":{"expr":{"ThemeDataColor":{"ColorId":0,"Percent":0}}}}},"show":{"expr":{"Literal":{"Value":"false"}}}}}],"background":[{"properties":{"show":{"expr":{"Literal":{"Value":"true"}}}}}],"border":[{"properties":{"show":{"expr":{"Literal":{"Value":"false"}}}}}],"padding":[{"properties":{"top":{"expr":{"Literal":{"Value":"0D"}}},"bottom":{"expr":{"Literal":{"Value":"0D"}}}}}]}}}',
     'filters': '[]',
     'height': 27.63,
     'width': 297.97,
     'x': 612.77,
     'y': 485.41,
     'z': 12000.0},
    {'config': '{"name":"3686aae53b2c71aca092","layouts":[{"id":0,"position":{"x":13.617021276595745,"y":123.79110251450676,"z":1000,"width":1006.42166344294,"height":590,"tabOrder":10000}}],"singleVisual":{"visualType":"shape","drillFilterOtherVisuals":true,"objects":{"shape":[{"properties":{"tileShape":{"expr":{"Literal":{"Value":"\'rectangle\'"}}},"roundEdge":{"expr":{"Literal":{"Value":"8L"}}}}}],"rotation":[{"properties":{"shapeAngle":{"expr":{"Literal":{"Value":"0L"}}}}}],"fill":[{"properties":{"fillColor":{"solid":{"color":{"expr":{"ThemeDataColor":{"ColorId":0,"Percent":0}}}}}},"selector":{"id":"default"}}],"outline":[{"properties":{"lineColor":{"solid":{"color":{"expr":{"ThemeDataColor":{"ColorId":2,"Percent":-0.5}}}}},"weight":{"expr":{"Literal":{"Value":"2D"}}}},"selector":{"id":"default"}}]},"vcObjects":{"background":[{"properties":{"transparency":{"expr":{"Literal":{"Value":"0D"}}},"color":{"solid":{"color":{"expr":{"ThemeDataColor":{"ColorId":0,"Percent":0}}}}}}}],"border":[{"properties":{}}],"general":[{"properties":{"keepLayerOrder":{"expr":{"Literal":{"Value":"true"}}}}}]}},"howCreated":"InsertVisualButton"}',
     'filters': '[]',
     'height': 590.0,
     'width': 1006.42,
     'x': 13.62,
     'y': 123.79,
     'z': 1000.0},
    {'config': '{"name":"53e60a0bad3b1870c63e","layouts":[{"id":0,"position":{"x":36,"y":132.16585265689855,"z":11000,"width":400,"height":72,"tabOrder":9000}}],"singleVisual":{"visualType":"slicer","projections":{"Values":[{"queryRef":"Capacities.Capacity_Name","active":true}]},"prototypeQuery":{"Version":2,"From":[{"Name":"c","Entity":"Capacities","Type":0}],"Select":[{"Column":{"Expression":{"SourceRef":{"Source":"c"}},"Property":"Capacity_Name"},"Name":"Capacities.Capacity_Name","NativeReferenceName":"Capacity Name"}]},"columnProperties":{"Capacities.Capacity_Name":{"displayName":"Capacity Name"}},"drillFilterOtherVisuals":true,"objects":{"data":[{"properties":{"mode":{"expr":{"Literal":{"Value":"\'Dropdown\'"}}}}}],"general":[{"properties":{"orientation":{"expr":{"Literal":{"Value":"1D"}}},"selfFilterEnabled":{"expr":{"Literal":{"Value":"true"}}}}}],"items":[{"properties":{"textSize":{"expr":{"Literal":{"Value":"12D"}}}}}],"header":[{"properties":{"textSize":{"expr":{"Literal":{"Value":"14D"}}}}}],"selection":[{"properties":{"singleSelect":{"expr":{"Literal":{"Value":"false"}}}}}]},"vcObjects":{"visualHeader":[{"properties":{"show":{"expr":{"Literal":{"Value":"true"}}}}}]}}}',
     'filters': '[]',
     'height': 72.0,
     'width': 400.0,
     'x': 36.0,
     'y': 132.17,
     'z': 11000.0},
    {'config': '{"name":"6f2e1e9f95e4fb32a377","layouts":[{"id":0,"position":{"x":460,"y":132,"z":10000,"width":550,"height":72,"tabOrder":8000}}],"singleVisual":{"visualType":"card","projections":{"Values":[{"queryRef":"Capacities.Single Capacity Selected"}]},"prototypeQuery":{"Version":2,"From":[{"Name":"c","Entity":"Capacities","Type":0}],"Select":[{"Measure":{"Expression":{"SourceRef":{"Source":"c"}},"Property":"Single Capacity Selected"},"Name":"Capacities.Single Capacity Selected","NativeReferenceName":"Single Capacity Selected"}]},"drillFilterOtherVisuals":true,"objects":{"categoryLabels":[{"properties":{"show":{"expr":{"Literal":{"Value":"false"}}}}}],"labels":[{"properties":{"fontSize":{"expr":{"Literal":{"Value":"16D"}}},"color":{"solid":{"color":{"expr":{"ThemeDataColor":{"ColorId":1,"Percent":0.6}}}}}}}]},"vcObjects":{"background":[{"properties":{"transparency":{"expr":{"Literal":{"Value":"0D"}}},"color":{"solid":{"color":{"expr":{"Measure":{"Expression":{"SourceRef":{"Entity":"Capacities"}},"Property":"Tansparent colour"}}}}}}}]}}}',
     'filters': '[]',
     'height': 72.0,
     'width': 550.0,
     'x': 460.0,
     'y': 132.0,
     'z': 10000.0},
    {'config': '{"name":"89a6011ae737ee39ad7a","layouts":[{"id":0,"position":{"x":1030.8936507238086,"y":270.3392440709288,"z":7000,"width":240.30155028527008,"height":443.3563602763233,"tabOrder":5000}}],"singleVisual":{"visualType":"textbox","drillFilterOtherVisuals":true,"objects":{"general":[{"properties":{"paragraphs":[{"textRuns":[{"value":"KPI definitions","textStyle":{"fontWeight":"bold","textDecoration":"underline"}},{"value":" -","textStyle":{"fontWeight":"bold"}}]},{"textRuns":[{"value":""}]},{"textRuns":[{"value":"PBI items ","textStyle":{"fontWeight":"bold"}},{"value":"include"},{"value":":","textStyle":{"fontWeight":"bold"}},{"value":" App, Dashboard, Datamart, Dataflow, Paginated Report, Report, Semantic models"}]},{"textRuns":[{"value":""}]},{"textRuns":[{"value":"Fabric items:","textStyle":{"fontWeight":"bold"}},{"value":" All remaining items except "},{"value":"PBI items","textStyle":{"fontWeight":"bold"}}]},{"textRuns":[{"value":""}]},{"textRuns":[{"value":"Large models:","textStyle":{"fontWeight":"bold"}},{"value":" Semantic models with storage mode set as "},{"value":"PremiumFiles","textStyle":{"fontWeight":"bold"}}]},{"textRuns":[{"value":""}]},{"textRuns":[{"value":"Cross region readiness:","textStyle":{"fontWeight":"bold"}}]},{"textRuns":[{"value":"\\t🔴 - Fabric item or Large model"}]},{"textRuns":[{"value":"\\t🟡 - Dataflow or Datamart"}]},{"textRuns":[{"value":"\\t🟢 - Good to move"}]},{"textRuns":[{"value":""}]}]}}]},"vcObjects":{"border":[{"properties":{"show":{"expr":{"Literal":{"Value":"true"}}},"color":{"solid":{"color":{"expr":{"ThemeDataColor":{"ColorId":2,"Percent":-0.5}}}}},"radius":{"expr":{"Literal":{"Value":"10D"}}},"width":{"expr":{"Literal":{"Value":"2D"}}}}}],"background":[{"properties":{"color":{"solid":{"color":{"expr":{"ThemeDataColor":{"ColorId":0,"Percent":0}}}}}}}]}}}',
     'filters': '[]',
     'height': 443.36,
     'width': 240.3,
     'x': 1030.89,
     'y': 270.34,
     'z': 7000.0},
    {'config': '{"name":"9c67315f6a6dc100ea50","layouts":[{"id":0,"position":{"x":37.628436152922056,"y":489.16966998798677,"z":8000,"width":397.9930746943679,"height":214.1926365627871,"tabOrder":6000}}],"singleVisual":{"visualType":"pivotTable","projections":{"Rows":[{"queryRef":"WorkspaceItems.Type","active":true},{"queryRef":"WorkspaceItems.Item_Name"}],"Values":[{"queryRef":"WorkspaceItems.Total Items"}]},"prototypeQuery":{"Version":2,"From":[{"Name":"w","Entity":"WorkspaceItems","Type":0}],"Select":[{"Column":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Type"},"Name":"WorkspaceItems.Type","NativeReferenceName":"Type"},{"Measure":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Total Items"},"Name":"WorkspaceItems.Total Items","NativeReferenceName":"Total Items"},{"Column":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Item_Name"},"Name":"WorkspaceItems.Item_Name","NativeReferenceName":"Item_Name"}]},"expansionStates":[{"roles":["Rows"],"levels":[{"queryRefs":["WorkspaceItems.Type"],"isCollapsed":true,"identityKeys":[{"Column":{"Expression":{"SourceRef":{"Entity":"WorkspaceItems"}},"Property":"Type"}}],"isPinned":true},{"queryRefs":["WorkspaceItems.Item_Name"],"isCollapsed":true,"isPinned":true}],"root":{"identityValues":null}}],"drillFilterOtherVisuals":true,"vcObjects":{"title":[{"properties":{"show":{"expr":{"Literal":{"Value":"true"}}},"text":{"expr":{"Literal":{"Value":"\'Workspace Items\'"}}}}}],"background":[{"properties":{"transparency":{"expr":{"Literal":{"Value":"100D"}}}}}]}}}',
     'filters': '[]',
     'height': 214.19,
     'width': 397.99,
     'x': 37.63,
     'y': 489.17,
     'z': 8000.0},
    {'config': '{"name":"9ca366c185006c4a97c8","layouts":[{"id":0,"position":{"x":36.97111631537861,"y":27.978142076502735,"z":3000,"width":394.6916471506636,"height":56.95550351288056,"tabOrder":1000}}],"singleVisual":{"visualType":"textbox","drillFilterOtherVisuals":true,"objects":{"general":[{"properties":{"paragraphs":[{"textRuns":[{"value":"Summary","textStyle":{"fontWeight":"bold","fontSize":"24pt"}}]}]}}]}}}',
     'filters': '[]',
     'height': 56.96,
     'width': 394.69,
     'x': 36.97,
     'y': 27.98,
     'z': 3000.0},
    {'config': '{"name":"b092367cb9c80420876d","layouts":[{"id":0,"position":{"x":1030.8936507238086,"y":9.612062011410803,"z":6000,"width":240.30155028527008,"height":249.91361229668087,"tabOrder":4000}}],"singleVisual":{"visualType":"textbox","drillFilterOtherVisuals":true,"objects":{"general":[{"properties":{"paragraphs":[{"textRuns":[{"value":"Report Summary","textStyle":{"fontWeight":"bold"}}]},{"textRuns":[{"value":"This report is to allow tenant/capacity admins to review their current state in terms of Capacities prior to transitioning from Premium to Fabric Capacities."}]},{"textRuns":[{"value":""}]},{"textRuns":[{"value":"Transition Scenarios","textStyle":{"fontWeight":"bold"}}]},{"textRuns":[{"value":"Same region"}],"listType":"bullet"},{"textRuns":[{"value":"Power BI Premium and Embedded supported"}],"listType":"bullet"},{"textRuns":[{"value":"Choose single, multi and all capacities"}],"listType":"bullet"}]}}]},"vcObjects":{"border":[{"properties":{"show":{"expr":{"Literal":{"Value":"true"}}},"radius":{"expr":{"Literal":{"Value":"11D"}}},"color":{"solid":{"color":{"expr":{"ThemeDataColor":{"ColorId":2,"Percent":-0.5}}}}},"width":{"expr":{"Literal":{"Value":"2D"}}}}}]}}}',
     'filters': '[]',
     'height': 249.91,
     'width': 240.3,
     'x': 1030.89,
     'y': 9.61,
     'z': 6000.0},
    {'config': '{"name":"c907bd0902a1828474bd","layouts":[{"id":0,"position":{"x":460,"y":132.16585265689855,"z":9000,"width":550,"height":72,"tabOrder":7000}}],"singleVisual":{"visualType":"multiRowCard","projections":{"Values":[{"queryRef":"Capacities.Region"},{"queryRef":"Capacities.Sku"},{"queryRef":"Workspaces.Total Workspaces"}]},"prototypeQuery":{"Version":2,"From":[{"Name":"c","Entity":"Capacities","Type":0},{"Name":"w","Entity":"Workspaces","Type":0}],"Select":[{"Column":{"Expression":{"SourceRef":{"Source":"c"}},"Property":"Region"},"Name":"Capacities.Region","NativeReferenceName":"Region"},{"Column":{"Expression":{"SourceRef":{"Source":"c"}},"Property":"Sku"},"Name":"Capacities.Sku","NativeReferenceName":"Sku"},{"Measure":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Total Workspaces"},"Name":"Workspaces.Total Workspaces","NativeReferenceName":"Total Workspaces"}],"OrderBy":[{"Direction":1,"Expression":{"Column":{"Expression":{"SourceRef":{"Source":"c"}},"Property":"Region"}}}]},"columnProperties":{"Capacities.Region":{"formatString":"G"}},"drillFilterOtherVisuals":true,"hasDefaultSort":true,"objects":{"categoryLabels":[{"properties":{"show":{"expr":{"Literal":{"Value":"true"}}}}}],"card":[{"properties":{"barShow":{"expr":{"Literal":{"Value":"true"}}}}}]},"vcObjects":{"lockAspect":[{"properties":{"show":{"expr":{"Literal":{"Value":"false"}}}}}],"general":[{"properties":{"keepLayerOrder":{"expr":{"Literal":{"Value":"false"}}}}}],"title":[{"properties":{"show":{"expr":{"Literal":{"Value":"true"}}},"text":{"expr":{"Aggregation":{"Expression":{"Column":{"Expression":{"SourceRef":{"Entity":"Capacities"}},"Property":"Capacity_Name"}},"Function":3}}},"titleWrap":{"expr":{"Literal":{"Value":"true"}}}}}],"divider":[{"properties":{"show":{"expr":{"Literal":{"Value":"false"}}}}}],"border":[{"properties":{"show":{"expr":{"Literal":{"Value":"false"}}}}}]}}}',
     'filters': '[]',
     'height': 72.0,
     'width': 550.0,
     'x': 460.0,
     'y': 132.17,
     'z': 9000.0},
    {'config': '{"name":"eeb0d5b9b6bc0f1f96a1","layouts":[{"id":0,"position":{"x":36.04523254279051,"y":227.0849650195802,"z":4000,"width":966.0122321467857,"height":255.92115105381262,"tabOrder":2000}}],"singleVisual":{"visualType":"pivotTable","projections":{"Rows":[{"queryRef":"Capacities.Capacity_Name","active":true},{"queryRef":"Workspaces.Name"}],"Values":[{"queryRef":"Workspaces.Total Workspaces"},{"queryRef":"WorkspaceItems.Total Items"},{"queryRef":"WorkspaceItems.Total PBI Items"},{"queryRef":"WorkspaceItems.Total Non PBI Items"},{"queryRef":"SemanticModels.Large Models"},{"queryRef":"Workspaces.Cross region readiness"},{"queryRef":"SemanticModels.DF + LSM"}]},"prototypeQuery":{"Version":2,"From":[{"Name":"w","Entity":"Workspaces","Type":0},{"Name":"w2","Entity":"WorkspaceItems","Type":0},{"Name":"s","Entity":"SemanticModels","Type":0},{"Name":"c","Entity":"Capacities","Type":0}],"Select":[{"Measure":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Total Workspaces"},"Name":"Workspaces.Total Workspaces","NativeReferenceName":"Total Workspaces"},{"Measure":{"Expression":{"SourceRef":{"Source":"w2"}},"Property":"Total Items"},"Name":"WorkspaceItems.Total Items","NativeReferenceName":"Total Items"},{"Column":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Name"},"Name":"Workspaces.Name","NativeReferenceName":"Name"},{"Measure":{"Expression":{"SourceRef":{"Source":"s"}},"Property":"Large Models"},"Name":"SemanticModels.Large Models","NativeReferenceName":"Large Models1"},{"Column":{"Expression":{"SourceRef":{"Source":"c"}},"Property":"Capacity_Name"},"Name":"Capacities.Capacity_Name","NativeReferenceName":"Capacity_Name"},{"Measure":{"Expression":{"SourceRef":{"Source":"w2"}},"Property":"Total PBI Items"},"Name":"WorkspaceItems.Total PBI Items","NativeReferenceName":"PBI Items"},{"Measure":{"Expression":{"SourceRef":{"Source":"w2"}},"Property":"Total Non PBI Items"},"Name":"WorkspaceItems.Total Non PBI Items","NativeReferenceName":"Fabric Items"},{"Measure":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Cross region readiness"},"Name":"Workspaces.Cross region readiness","NativeReferenceName":"Cross region readiness"},{"Measure":{"Expression":{"SourceRef":{"Source":"s"}},"Property":"DF + LSM"},"Name":"SemanticModels.DF + LSM","NativeReferenceName":"DF + LSM"}],"OrderBy":[{"Direction":1,"Expression":{"Column":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Name"}}}]},"expansionStates":[{"roles":["Rows"],"levels":[{"queryRefs":["Capacities.Capacity_Name"],"isCollapsed":true,"identityKeys":[{"Column":{"Expression":{"SourceRef":{"Entity":"Capacities"}},"Property":"Capacity_Name"}}],"isPinned":true},{"queryRefs":["Workspaces.Name"],"isCollapsed":true,"identityKeys":[{"Column":{"Expression":{"SourceRef":{"Entity":"Workspaces"}},"Property":"Name"}}],"isPinned":true}],"root":{"identityValues":null}}],"columnProperties":{"WorkspaceItems.Total PBI Items":{"displayName":"PBI Items"},"WorkspaceItems.Total Non PBI Items":{"displayName":"Fabric Items"},"Workspaces.Cross region readiness":{"displayName":"Cross region readiness"}},"queryOptions":{"keepProjectionOrder":true},"drillFilterOtherVisuals":true,"objects":{"values":[{"properties":{},"selector":{"data":[{"dataViewWildcard":{"matchingOption":1}}],"metadata":"SemanticModels.DF + LSM"}},{"properties":{},"selector":{"data":[{"dataViewWildcard":{"matchingOption":1}}],"metadata":"Workspaces.Cross region readiness"}},{"properties":{"icon":{"kind":"Icon","layout":{"expr":{"Literal":{"Value":"\'IconOnly\'"}}},"verticalAlignment":{"expr":{"Literal":{"Value":"\'Middle\'"}}},"value":{"expr":{"Conditional":{"Cases":[{"Condition":{"Comparison":{"ComparisonKind":0,"Left":{"Measure":{"Expression":{"SourceRef":{"Entity":"Workspaces"}},"Property":"Cross region readiness"}},"Right":{"Literal":{"Value":"\'Good to move\'"}}},"Annotations":{"PowerBI.SQExprEvaluationKind":1,"PowerBI.SQExprTextOperatorOption":2}},"Value":{"Literal":{"Value":"\'CircleHigh\'"}}},{"Condition":{"Comparison":{"ComparisonKind":0,"Left":{"Measure":{"Expression":{"SourceRef":{"Entity":"Workspaces"}},"Property":"Cross region readiness"}},"Right":{"Literal":{"Value":"\'Dataflow or Datamart\'"}}},"Annotations":{"PowerBI.SQExprEvaluationKind":1,"PowerBI.SQExprTextOperatorOption":2}},"Value":{"Literal":{"Value":"\'CircleMedium\'"}}},{"Condition":{"Comparison":{"ComparisonKind":0,"Left":{"Measure":{"Expression":{"SourceRef":{"Entity":"Workspaces"}},"Property":"Cross region readiness"}},"Right":{"Literal":{"Value":"\'Fabric item or Large model\'"}}},"Annotations":{"PowerBI.SQExprEvaluationKind":1,"PowerBI.SQExprTextOperatorOption":2}},"Value":{"Literal":{"Value":"\'CircleLow\'"}}}]}}}}},"selector":{"data":[{"dataViewWildcard":{"matchingOption":0}}],"metadata":"Workspaces.Cross region readiness"}},{"properties":{"icon":{"kind":"Icon","layout":{"expr":{"Literal":{"Value":"\'IconOnly\'"}}},"verticalAlignment":{"expr":{"Literal":{"Value":"\'Top\'"}}},"value":{"expr":{"Conditional":{"Cases":[{"Condition":{"Comparison":{"ComparisonKind":0,"Left":{"Measure":{"Expression":{"SourceRef":{"Entity":"SemanticModels"}},"Property":"DF + LSM"}},"Right":{"Literal":{"Value":"\'Y\'"}}},"Annotations":{"PowerBI.SQExprEvaluationKind":1,"PowerBI.SQExprTextOperatorOption":2}},"Value":{"Literal":{"Value":"\'SignMedium\'"}}}]}}}}},"selector":{"data":[{"dataViewWildcard":{"matchingOption":0}}],"metadata":"SemanticModels.DF + LSM"}}],"columnFormatting":[{"properties":{"alignment":{"expr":{"Literal":{"Value":"\'Center\'"}}},"styleTotal":{"expr":{"Literal":{"Value":"true"}}}},"selector":{"metadata":"Workspaces.Cross region readiness"}},{"properties":{"styleTotal":{"expr":{"Literal":{"Value":"false"}}},"styleHeader":{"expr":{"Literal":{"Value":"false"}}},"alignment":{"expr":{"Literal":{"Value":"\'Center\'"}}}},"selector":{"metadata":"SemanticModels.DF + LSM"}}],"columnWidth":[{"properties":{"value":{"expr":{"Literal":{"Value":"300.9267158627982D"}}}},"selector":{"metadata":"Capacities.Capacity_Name"}}]},"vcObjects":{"title":[{"properties":{"show":{"expr":{"Literal":{"Value":"true"}}},"text":{"expr":{"Literal":{"Value":"\'Capacities\'"}}},"fontSize":{"expr":{"Literal":{"Value":"14D"}}}}}],"background":[{"properties":{"transparency":{"expr":{"Literal":{"Value":"100D"}}}}}]}}}',
     'filters': '[]',
     'height': 255.92,
     'width': 966.01,
     'x': 36.05,
     'y': 227.08,
     'z': 4000.0},
    {'config': '{"name":"eec2319a0fbda582c675","layouts":[{"id":0,"position":{"x":431.6627634660422,"y":20,"z":2000,"width":578.5480093676815,"height":80,"tabOrder":0}}],"singleVisual":{"visualType":"multiRowCard","projections":{"Values":[{"queryRef":"Capacities.Total Capacities"},{"queryRef":"Workspaces.Total Workspaces"},{"queryRef":"WorkspaceItems.Total Items"}]},"prototypeQuery":{"Version":2,"From":[{"Name":"c","Entity":"Capacities","Type":0},{"Name":"w","Entity":"Workspaces","Type":0},{"Name":"w1","Entity":"WorkspaceItems","Type":0}],"Select":[{"Measure":{"Expression":{"SourceRef":{"Source":"c"}},"Property":"Total Capacities"},"Name":"Capacities.Total Capacities","NativeReferenceName":"Total Capacities"},{"Measure":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Total Workspaces"},"Name":"Workspaces.Total Workspaces","NativeReferenceName":"Total Workspaces"},{"Measure":{"Expression":{"SourceRef":{"Source":"w1"}},"Property":"Total Items"},"Name":"WorkspaceItems.Total Items","NativeReferenceName":"Total Items"}],"OrderBy":[{"Direction":2,"Expression":{"Measure":{"Expression":{"SourceRef":{"Source":"c"}},"Property":"Total Capacities"}}}]},"drillFilterOtherVisuals":true,"hasDefaultSort":true,"objects":{"card":[{"properties":{"outlineStyle":{"expr":{"Literal":{"Value":"0D"}}}}}],"dataLabels":[{"properties":{"fontSize":{"expr":{"Literal":{"Value":"20D"}}},"bold":{"expr":{"Literal":{"Value":"true"}}}}}],"categoryLabels":[{"properties":{"fontSize":{"expr":{"Literal":{"Value":"14D"}}}}}]},"vcObjects":{"title":[{"properties":{}}],"visualHeader":[{"properties":{}}],"border":[{"properties":{}}]}}}',
     'filters': '[]',
     'height': 80.0,
     'width': 578.55,
     'x': 431.66,
     'y': 20.0,
     'z': 2000.0},
    {'config': '{"name":"f10f7af3c09f8a17a196","layouts":[{"id":0,"position":{"x":13.500082397963855,"y":10.500064087305221,"z":0,"width":1006.5061432259718,"height":102.00062256239356,"tabOrder":11000}}],"singleVisual":{"visualType":"shape","drillFilterOtherVisuals":true,"objects":{"shape":[{"properties":{"tileShape":{"expr":{"Literal":{"Value":"\'rectangle\'"}}},"roundEdge":{"expr":{"Literal":{"Value":"8L"}}}}}],"rotation":[{"properties":{"shapeAngle":{"expr":{"Literal":{"Value":"0L"}}}}}],"fill":[{"properties":{"fillColor":{"solid":{"color":{"expr":{"ThemeDataColor":{"ColorId":0,"Percent":0}}}}}},"selector":{"id":"default"}}],"outline":[{"properties":{"lineColor":{"solid":{"color":{"expr":{"ThemeDataColor":{"ColorId":2,"Percent":-0.5}}}}},"weight":{"expr":{"Literal":{"Value":"2D"}}}},"selector":{"id":"default"}}]},"vcObjects":{"background":[{"properties":{"transparency":{"expr":{"Literal":{"Value":"0D"}}},"color":{"solid":{"color":{"expr":{"ThemeDataColor":{"ColorId":0,"Percent":0}}}}}}}],"border":[{"properties":{}}],"general":[{"properties":{"keepLayerOrder":{"expr":{"Literal":{"Value":"true"}}}}}]}},"howCreated":"InsertVisualButton"}',
     'filters': '[]',
     'height': 102.0,
     'width': 1006.51,
     'x': 13.5,
     'y': 10.5,
     'z': 0.0}],
   'width': 1280.0},
  {'config': '{}',
   'displayName': 'Workspace Items',
   'displayOption': 1,
   'filters': '[]',
   'height': 720.0,
   'name': 'd889f7d26043cef4c7a8',
   'ordinal': 1,
   'visualContainers': [{'config': '{"name":"4fd592453c9bc272ee20","layouts":[{"id":0,"position":{"x":838.6524104955926,"y":16.821108519968906,"z":4000,"width":427.73675950778073,"height":201.85330223962686,"tabOrder":4000}}],"singleVisual":{"visualType":"slicer","projections":{"Values":[{"queryRef":"WorkspaceItems.Item_Name","active":true}]},"prototypeQuery":{"Version":2,"From":[{"Name":"w","Entity":"WorkspaceItems","Type":0}],"Select":[{"Column":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Item_Name"},"Name":"WorkspaceItems.Item_Name","NativeReferenceName":"Item_Name"}]},"drillFilterOtherVisuals":true,"objects":{"data":[{"properties":{"mode":{"expr":{"Literal":{"Value":"\'Basic\'"}}}}}],"general":[{"properties":{"selfFilterEnabled":{"expr":{"Literal":{"Value":"true"}}}}}]}}}',
     'filters': '[]',
     'height': 201.85,
     'width': 427.74,
     'x': 838.65,
     'y': 16.82,
     'z': 4000.0},
    {'config': '{"name":"6607ffa5518454f810ae","layouts":[{"id":0,"position":{"x":630.791569498834,"y":16.821108519968906,"z":3000,"width":199.45028673677416,"height":201.85330223962686,"tabOrder":3000}}],"singleVisual":{"visualType":"slicer","projections":{"Values":[{"queryRef":"WorkspaceItems.Type","active":true}]},"prototypeQuery":{"Version":2,"From":[{"Name":"w","Entity":"WorkspaceItems","Type":0}],"Select":[{"Column":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Type"},"Name":"WorkspaceItems.Type","NativeReferenceName":"Type"}]},"drillFilterOtherVisuals":true,"objects":{"data":[{"properties":{"mode":{"expr":{"Literal":{"Value":"\'Basic\'"}}}}}],"general":[{"properties":{"selfFilterEnabled":{"expr":{"Literal":{"Value":"true"}}}}}]}}}',
     'filters': '[]',
     'height': 201.85,
     'width': 199.45,
     'x': 630.79,
     'y': 16.82,
     'z': 3000.0},
    {'config': '{"name":"83de2a713a395db23f22","layouts":[{"id":0,"position":{"x":322.0040773822619,"y":16.821108519968906,"z":2000,"width":300.3769378565876,"height":201.85330223962686,"tabOrder":2000}}],"singleVisual":{"visualType":"slicer","projections":{"Values":[{"queryRef":"Workspaces.Name","active":true}]},"prototypeQuery":{"Version":2,"From":[{"Name":"w","Entity":"Workspaces","Type":0}],"Select":[{"Column":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Name"},"Name":"Workspaces.Name","NativeReferenceName":"Workspace Name"}]},"columnProperties":{"Workspaces.Name":{"displayName":"Workspace Name"}},"drillFilterOtherVisuals":true,"objects":{"data":[{"properties":{"mode":{"expr":{"Literal":{"Value":"\'Basic\'"}}}}}],"general":[{"properties":{"selfFilterEnabled":{"expr":{"Literal":{"Value":"true"}}}}}]}}}',
     'filters': '[{"name":"62438d0a1f6e0f7ade63","expression":{"Column":{"Expression":{"SourceRef":{"Entity":"Workspaces"}},"Property":"Name"}},"filter":{"Version":2,"From":[{"Name":"w","Entity":"Workspaces","Type":0}],"Where":[{"Condition":{"Not":{"Expression":{"In":{"Expressions":[{"Column":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Name"}}],"Values":[[{"Literal":{"Value":"null"}}]]}}}}}]},"type":"Categorical","howCreated":0,"objects":{"general":[{"properties":{"isInvertedSelectionMode":{"expr":{"Literal":{"Value":"true"}}}}}]},"isHiddenInViewMode":true,"isLockedInViewMode":true}]',
     'height': 201.85,
     'width': 300.38,
     'x': 322.0,
     'y': 16.82,
     'z': 2000.0},
    {'config': '{"name":"be0d6e871f1600ca5d43","layouts":[{"id":0,"position":{"x":12.015077514263504,"y":240.30155028527008,"z":0,"width":1254.37409248911,"height":463.78199205057126,"tabOrder":0}}],"singleVisual":{"visualType":"tableEx","projections":{"Values":[{"queryRef":"Workspaces.Name"},{"queryRef":"WorkspaceItems.Item_Name"},{"queryRef":"WorkspaceItems.Type"},{"queryRef":"WorkspaceItems.Creator_Principal_Display_Name"},{"queryRef":"WorkspaceItems.Creator_User_Principal_Name"},{"queryRef":"WorkspaceItems.Description"},{"queryRef":"WorkspaceItems.Item_Id"},{"queryRef":"WorkspaceItems.Workspace_Id"},{"queryRef":"WorkspaceItems.Capacity_Id"}]},"prototypeQuery":{"Version":2,"From":[{"Name":"w","Entity":"WorkspaceItems","Type":0},{"Name":"w1","Entity":"Workspaces","Type":0}],"Select":[{"Column":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Creator_Principal_Display_Name"},"Name":"WorkspaceItems.Creator_Principal_Display_Name","NativeReferenceName":"Creater Name"},{"Column":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Creator_User_Principal_Name"},"Name":"WorkspaceItems.Creator_User_Principal_Name","NativeReferenceName":"Creator_User_Principal_Name"},{"Column":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Description"},"Name":"WorkspaceItems.Description","NativeReferenceName":"Description"},{"Column":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Item_Id"},"Name":"WorkspaceItems.Item_Id","NativeReferenceName":"Item_Id"},{"Column":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Type"},"Name":"WorkspaceItems.Type","NativeReferenceName":"Type"},{"Column":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Item_Name"},"Name":"WorkspaceItems.Item_Name","NativeReferenceName":"Item Name"},{"Column":{"Expression":{"SourceRef":{"Source":"w1"}},"Property":"Name"},"Name":"Workspaces.Name","NativeReferenceName":"Workspace Name"},{"Column":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Capacity_Id"},"Name":"WorkspaceItems.Capacity_Id","NativeReferenceName":"Capacity_Id"},{"Column":{"Expression":{"SourceRef":{"Source":"w"}},"Property":"Workspace_Id"},"Name":"WorkspaceItems.Workspace_Id","NativeReferenceName":"Workspace_Id"}]},"columnProperties":{"Workspaces.Name":{"displayName":"Workspace Name"},"WorkspaceItems.Item_Name":{"displayName":"Item Name"},"WorkspaceItems.Creator_Principal_Display_Name":{"displayName":"Creater Name"}},"drillFilterOtherVisuals":true,"objects":{"columnWidth":[{"properties":{"value":{"expr":{"Literal":{"Value":"152.03015640527124D"}}}},"selector":{"metadata":"Workspaces.Name"}},{"properties":{"value":{"expr":{"Literal":{"Value":"257.85126561932225D"}}}},"selector":{"metadata":"WorkspaceItems.Item_Name"}},{"properties":{"value":{"expr":{"Literal":{"Value":"116.21658267794601D"}}}},"selector":{"metadata":"WorkspaceItems.Type"}},{"properties":{"value":{"expr":{"Literal":{"Value":"141.01508041716662D"}}}},"selector":{"metadata":"WorkspaceItems.Creator_Principal_Display_Name"}},{"properties":{"value":{"expr":{"Literal":{"Value":"143.657293700058D"}}}},"selector":{"metadata":"WorkspaceItems.Description"}}]}}}',
     'filters': '[]',
     'height': 463.78,
     'width': 1254.37,
     'x': 12.02,
     'y': 240.3,
     'z': 0.0},
    {'config': '{"name":"be0f648121b687704783","layouts":[{"id":0,"position":{"x":12.015077514263504,"y":16.821108519968906,"z":1000,"width":300,"height":201.85330223962686,"tabOrder":1000}}],"singleVisual":{"visualType":"slicer","projections":{"Values":[{"queryRef":"Capacities.Capacity_Name","active":true}]},"prototypeQuery":{"Version":2,"From":[{"Name":"c","Entity":"Capacities","Type":0}],"Select":[{"Column":{"Expression":{"SourceRef":{"Source":"c"}},"Property":"Capacity_Name"},"Name":"Capacities.Capacity_Name","NativeReferenceName":"Capacity_Name"}]},"drillFilterOtherVisuals":true,"objects":{"data":[{"properties":{"mode":{"expr":{"Literal":{"Value":"\'Basic\'"}}}}}],"general":[{"properties":{"selfFilterEnabled":{"expr":{"Literal":{"Value":"true"}}}}}]}}}',
     'filters': '[]',
     'height': 201.85,
     'width': 300.0,
     'x': 12.02,
     'y': 16.82,
     'z': 1000.0}],
   'width': 1280.0}]}

##### Setting up and Installing discovery report

In [ ]:
df_items =fabric.list_items()
lakehouse_exists = df_items['Display Name'].eq(lakehouse).any()

if lakehouse_exists:
    print("Lakehouse already exists.")
    lhid = ( df_items[df_items['Display Name'].eq(lakehouse)&df_items['Type'].eq('Lakehouse')].iloc[0, 0])
else:
    print("Lakehouse does not exist. Creating...")
    lhid = fabric.create_lakehouse(lakehouse)

all_workspaces_df = labs_admin.list_workspaces()
all_workspaces_df = all_workspaces_df[["Id", "Name", "State", "Type", "Capacity Id"]]
all_workspaces_df['Capacity Id'] = all_workspaces_df['Capacity Id'].fillna("-1")
labs.save_as_delta_table(dataframe=all_workspaces_df, delta_table_name="Workspaces", write_mode="overwrite",lakehouse=lakehouse)

df_capacities = labs_admin.list_capacities()
# df_capacities = labs_admin._capacities._list_capacities_meta()
new_record = pd.DataFrame([{
    "Capacity Id": "-1",
    "Capacity Name": "Non Premium",
    "Sku": "N/A",
    "Region": "N/A",
    "State": "Active",
    "Admins": [],
    "Users": [""]
}])

df_capacities = pd.concat([df_capacities, new_record], ignore_index=True)
# display(df_capacities)
labs.save_as_delta_table(dataframe=df_capacities,delta_table_name="Capacities",write_mode="overwrite",lakehouse=lakehouse)

df_items = labs_admin.list_items()
labs.save_as_delta_table(dataframe=df_items,delta_table_name="WorkspaceItems",write_mode="overwrite",lakehouse=lakehouse)

df_semantic_models = labs_admin.list_datasets()
df_semantic_models["Upstream Datasets"] = df_semantic_models["Upstream Datasets"].astype(str)
df_semantic_models["Users"] = df_semantic_models["Users"].astype(str)
labs.save_as_delta_table(dataframe=df_semantic_models,delta_table_name="SemanticModels",write_mode="overwrite",lakehouse=lakehouse)

try:
    labs.create_semantic_model_from_bim(
    dataset=semantic_model_name,
    bim_file=bimjson)
except ValueError as e:
    
    this_wn, this_wid = fabric.resolve_workspace_name_and_id()

    # Check if the specific error message matches
    if f"The '{semantic_model_name}' semantic model already exists as a semantic model in the '{this_wn}' workspace." in str(e):
        print(f"{labs._helper_functions.icons.yellow_dot} The '{semantic_model_name}' semantic model may already exist in the '{this_wn}' workspace. The Model was not updated!")
        pass
    else:
        # Re-raise the exception if it's not the specific error you're looking for
        raise
except Exception as e:
    print(e)

In [ ]:
import time

check_interval = 5  # Interval in seconds between checks
max_retries = 12  # Maximum number of retries (e.g., 12 retries with 5 seconds interval = 1 minute)

# Loop to wait until there is a record in the DataFrame or max retries are reached
retries = 0
while retries < max_retries:
    df = fabric.list_datasets()
    df_filt = df[df["Dataset Name"] == semantic_model_name]
    
    if not df_filt.empty:
        labs.directlake.update_direct_lake_model_lakehouse_connection(dataset=semantic_model_name,lakehouse=lakehouse)

        labs_report.create_report_from_reportjson(
            report=report_name,
            dataset=semantic_model_name,
            report_json=reportjson
            )
        print('Report and Semantic Model created in current workspace.')
        break
    
    print(f"No model found. Retrying in {check_interval} seconds... ({retries + 1}/{max_retries})")
    time.sleep(check_interval)
    retries += 1

if retries == max_retries:
    print("Max retries reached. No semantic model found.")